In [112]:

import pickle
import pandas as pd
import numpy as np
import torch

import sys
sys.path.append("..")
from cd_zoo.tools.scoring_tools import score

In [155]:
out_path1 = "/home/datasets4/stein/robust_exp/release_ensemble_results/test_corrected_INSTsmall_wcg_results.p"
out_path2 = "/home/datasets4/stein/robust_exp/release_ensemble_results/test_corrected_INSTsmall_inst_results.p"

In [156]:
(X,out_Y,out_meta,method_ordering) = pickle.load(open(out_path1, "rb"))
(X_inst,out_Y_inst,out_meta_inst,method_ordering_inst) = pickle.load(open(out_path2, "rb"))

In [157]:
assert np.sum(out_meta_inst != out_meta).sum() == 0, "Something off with the meta tables WCG and INST"

In [175]:
%%capture
ds = out_meta["ds"].unique()

selection = out_meta[out_meta["ds"] == ds[0]]

runs = selection["run"].unique()
stack = []
for x in runs:
    samples = selection[selection["run"] == x]

    assert len(samples) == 100, "Expected 100 samples per run, but got {}".format(len(samples))

    pred = X[samples.index][:,-1]
    target = out_Y[samples.index]
    pred_inst = X_inst[samples.index][:,-1]
    target_inst = out_Y_inst[samples.index]
    if samples["no_inst"].sum() == 0:
        print("with inst")
        stack.append(score(target,pred, instant_labs=target_inst, instant_preds=pred_inst, per_sample_metrics=True, verbose=0))
    else: 
        stack.append(score(target,pred, per_sample_metrics=True, verbose=0))

In [177]:
performance = [x.loc["SHD individual"][["SG_max","WCG"]].values for x in stack]
# This should be the same number as we report for the table in the paper.
mean_performance = np.stack(performance,axis=0).mean(axis=0)

In [1]:
import pickle
a = pickle.load(open("/home/datasets4/stein/robust_exp/ensemble_experiment/random_5_tcd_arena.p", "rb"))

In [6]:
a[0][0].shape

(7029, 5)

In [11]:
import pickle
b = pickle.load(open("/home/datasets4/stein/robust_exp/release_ensemble_rivers/releease_random_5_tcd_arena.p", "rb"))

array([[ 0.29530632, 38.02400038, 42.73136693, 43.9739635 , 39.51718191],
       [ 0.29530632, 42.90231755, 44.99077782, 41.10454899, 37.4203328 ],
       [ 0.35948489, 46.25723344, 43.21350425, 40.48061315, 37.14793636],
       ...,
       [ 0.51612049, 50.52948483, 41.66271204, 32.17993634, 29.4236011 ],
       [ 0.51612049, 48.11248568, 36.91697668, 28.83726261, 26.84773079],
       [ 0.46937556, 47.43004733, 35.1596571 , 27.97931486, 26.32441644]])

In [ ]:
pred = X[samples.index][:,0]
target = out_Y[samples.index]


stack = score(target,preds per_sample_metrics=True, verbose=0)

In [ ]:
stack = score(pred,target, per_sample_metrics=True, verbose=0)

,SG_max,WCG,SG_mean
Metric,,,
AUROC Joint,0.554337,0.592236,0.555910
Max F1 Joint,0.361193,0.251982,0.361193
Acc Joint,0.7796,0.921467,0.779600
SHD Joint,1.0,1.0,1.000000
Max F1 th Joint,False,True,1.000000
Acc th Joint,1.0,1.0,1.000000
SHD th Joint,1.0,1.0,1.000000
Null AUROC Joint,0.5,0.5,0.500000
NULL F1 Joint,0.361193,0.14563,0.361193


In [26]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [73]:
from dl_components.linear import SimpleLinear
from dl_components.mlp import SimpleMLP
from dl_components.transformer import SimpleTransformer

In [74]:
random_tensor = torch.rand(16,6,7,7,4) 

In [78]:
net = SimpleLinear(methods= 10, lag_dimension=3, n_vars=7)
net = SimpleMLP( methods= 6, lag_dimension=4, n_vars=7, hidden_dims=[32, 64],dropout_rate=0.1)
net = SimpleTransformer(methods= 6, lag_dimension=4, n_vars=7, model_dim= 256, num_heads=4, num_layers=2, dropout=0.1, pos_embedding_dim=16)

In [79]:
net(random_tensor).shape

torch.Size([16, 7, 7, 4])

In [23]:
raw = pickle.load(open(out_path + b, "rb"))

In [32]:
raw[1].shape

(6000, 5, 5, 3)